[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/greends-pml/blob/main/notebooks/T8_torch_NN_pipeline_and_questions_for_assign_3.ipynb)

# Pipeline for deep learning with PyTorch

| Step | Key Actions | Main PyTorch Classes/Methods |
|------|-------------|------------------------------|
| Data Preparation | Load, transform, batch, and split data | `torch.utils.data.Dataset`, `DataLoader`, `torchvision.transforms` |
| Model Development | Define architecture, choose loss/optimizer, set hyperparameters | `torch.nn.Module`, `torch.nn.Parameter`, `torch.nn.functional`, `torch.optim` |
| Model Training | Forward pass, loss computation, backward pass, parameter update, epochs | `forward()`, `loss.backward()`, `optimizer.step()`, `optimizer.zero_grad()`, `model.train()` |
| Validation | Evaluate on validation set, compute metrics, tune hyperparameters | `model.eval()`, `torch.no_grad()`, metric functions |
| Testing/Deployment | Final evaluation on test set, save and deploy model | `torch.save()`, `torch.load()`, `model.eval()`, `torch.jit` |

## Step 1: Data Preparation

In [ ]:
!pip install torchvision

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import time

In [ ]:
# Define a transformation to flatten the 8x8 images
transform = transforms.Compose([
    transforms.Resize((8, 8)),                        # Resize to 8x8
    transforms.ToTensor(),                            # Convert to tensor
    transforms.Lambda(lambda x: x.view(-1))           # Flatten
])

# Load the MNIST dataset
train_dataset = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Create DataLoaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

---
## ✅ QUESTION 1 — DataLoaders

**Q1a. What is the type of objects yielded by the train and test dataloaders?**

Each dataloader yields a **tuple of two `torch.Tensor` objects**: `(images, labels)`.
- `images` is a FloatTensor of shape `[batch_size, 64]`
- `labels` is a LongTensor of shape `[batch_size]`

**Q1b. What is the shape of images returned by `for images, labels in train_loader`? How do you interpret that?**

Shape is **`[64, 64]`**:
- First `64` = batch size (64 images per batch)
- Second `64` = flattened 8×8 image (64 pixels per image)

**Q1c. Why does train use `shuffle=True` but test uses `shuffle=False`?**

- `shuffle=True` for **training**: prevents the model from learning the order of examples, which would cause order-dependent bias and hurt generalisation.
- `shuffle=False` for **testing**: order doesn't affect evaluation metrics, and keeping it consistent makes results reproducible across runs.

In [ ]:
# Verify Q1a and Q1b programmatically
images, labels = next(iter(train_loader))
print(f"Type of batch     : {type((images, labels))}")
print(f"Type of images    : {type(images)}")
print(f"Type of labels    : {type(labels)}")
print(f"Shape of images   : {images.shape}  -> [batch_size=64, pixels=8x8=64]")
print(f"Shape of labels   : {labels.shape}  -> [batch_size=64]")

In [ ]:
# Visualize some examples
images_vis, labels_vis = zip(*[train_dataset[i] for i in range(12)])

fig, axes = plt.subplots(3, 4, figsize=(9, 12))
for i, ax in enumerate(axes.flat):
    img = images_vis[i].reshape(8, 8)
    ax.imshow(img, cmap='gray')
    ax.set_title(str(labels_vis[i]), fontsize=12)
    ax.axis('off')
plt.tight_layout()
plt.show()

---
## Step 2: Model Development

In [ ]:
# Original model (1 hidden layer) — kept for reference
class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

input_size  = 8 * 8   # 64 pixels (8x8 resized images)
hidden_size = 128
num_classes = 10

model = SimpleNN(input_size, hidden_size, num_classes)
print(model)

---
## ✅ QUESTION 2 — Model Architecture

**Q2a. Two changes needed to use original 28×28 images instead of 8×8:**

1. **Remove** `transforms.Resize((8, 8))` from the transform (or change it to `transforms.Resize((28, 28))`, but since MNIST is natively 28×28, just remove it).
2. **Change** `input_size = 8 * 8` → `input_size = 28 * 28` (= 784) when instantiating the model.

**Q2b. Change to `SimpleNN` to add a second hidden layer of size 128:** — see `TwoLayerNN` class below.

**Q2c. What is a ReLU activation function (`nn.ReLU`)?**

ReLU stands for **Rectified Linear Unit**. It is defined as:

`f(x) = max(0, x)`

It outputs the input value if positive, and zero otherwise. It is computationally cheap and avoids the vanishing gradient problem that affects sigmoid/tanh.

**Q2d. Why are non-linear activation functions like ReLU necessary for deep learning?**

Without non-linear activations, stacking multiple linear layers is mathematically equivalent to a **single linear transformation** (since the composition of linear functions is still linear). The network would only be able to learn linear decision boundaries regardless of depth. Non-linear activations like ReLU allow the network to learn **complex, non-linear mappings** from inputs to outputs — which is essential for real-world tasks like image recognition.

In [ ]:
# prompt: Modify the SimpleNN class to add a second hidden layer of size 128,
# with ReLU activation after each hidden layer.
# Modification: renamed layers fc1->fc1, fc2->fc2 (new hidden), fc3->output
# to keep naming consistent and clear.

class TwoLayerNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(TwoLayerNN, self).__init__()
        self.fc1   = nn.Linear(input_size, hidden_size)   # 1st hidden layer
        self.relu1 = nn.ReLU()
        self.fc2   = nn.Linear(hidden_size, hidden_size)  # 2nd hidden layer (NEW)
        self.relu2 = nn.ReLU()                            # activation for 2nd layer (NEW)
        self.fc3   = nn.Linear(hidden_size, num_classes)  # output layer

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu1(out)
        out = self.fc2(out)   # NEW
        out = self.relu2(out) # NEW
        out = self.fc3(out)
        return out

model2 = TwoLayerNN(input_size, hidden_size, num_classes)
print(model2)

In [ ]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

---
## Step 3: Model Training (baseline)

In [ ]:
num_epochs = 3

for epoch in range(num_epochs):
    print(f'epoch: {epoch+1}; time: {round(time.time())}')
    model.train()
    for images, labels in train_loader:
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

---
## ✅ QUESTION 3 — num_workers and Training Time

**Q: Does processing time decrease if you add `num_workers=2` when defining the dataloader?**

The `num_workers` parameter controls how many CPU subprocesses are used to load data in parallel while the GPU trains. With `num_workers=0` (default), data loading is done synchronously in the main process. With `num_workers=2`, two worker processes prefetch batches concurrently.

**Expected result on Colab with GPU:** The improvement is often small or negligible for this small dataset (8×8 MNIST). With larger datasets and heavier augmentations, `num_workers>0` gives a clear speedup.

In [ ]:
# prompt: Compare training time with and without num_workers=2 in DataLoader
# to determine if parallel data loading speeds up training.
# Modification: Used the same model and 3 epochs for a fair comparison;
# re-initialize optimizer before each run so weights start from same state.

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def train_and_time(loader, label):
    """Re-initialize model & optimizer, train for 3 epochs, return elapsed time."""
    m = SimpleNN(input_size, hidden_size, num_classes).to(device)
    opt = optim.Adam(m.parameters(), lr=0.001)
    start = time.time()
    for epoch in range(3):
        m.train()
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            out = m(imgs)
            loss = criterion(out, lbls)
            opt.zero_grad()
            loss.backward()
            opt.step()
    elapsed = time.time() - start
    print(f"{label}: {elapsed:.2f}s")
    return elapsed

# Without num_workers
loader_0w = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
t0 = train_and_time(loader_0w, "No workers (num_workers=0)")

# With num_workers=2
loader_2w = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
t2 = train_and_time(loader_2w, "With num_workers=2  ")

print(f"\nSpeedup factor: {t0/t2:.2f}x")
if t2 < t0:
    print("✅ num_workers=2 is FASTER")
else:
    print("⚠️  num_workers=2 is NOT faster for this small dataset — overhead of spawning workers outweighs the benefit.")

---
## Change device: run training on GPU

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model.to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(num_epochs):
    print(f'epoch: {epoch+1}; time: {round(time.time())}')
    model.train()
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

---
## Step 4: Validation — with Bug Fix

In [ ]:
def compute_accuracy(outputs, labels):
    _, predicted = torch.max(outputs.data, 1)
    correct = (predicted == labels).sum().item()
    return correct / len(labels)

### Original (buggy) training loop — shown for comparison

In [ ]:
# Original loop (BUGGY): train accuracy is computed DURING training (model still updating)
# This causes train accuracy to appear lower than it really is.

num_epochs = 5
history_buggy = {'epoch': [], 'train_accuracy': [], 'val_accuracy': []}

model_buggy = SimpleNN(input_size, hidden_size, num_classes).to(device)
optimizer_buggy = optim.Adam(model_buggy.parameters(), lr=0.001)

for epoch in range(num_epochs):
    model_buggy.train()
    batch_accuracies = []
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model_buggy(images)
        loss = criterion(outputs, labels)
        optimizer_buggy.zero_grad()
        loss.backward()
        optimizer_buggy.step()
        batch_accuracies.append(compute_accuracy(outputs, labels))  # <-- measured mid-training
    history_buggy['epoch'].append(epoch)
    history_buggy['train_accuracy'].append(sum(batch_accuracies) / len(batch_accuracies))

    model_buggy.eval()
    batch_accuracies = []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_buggy(images)
            batch_accuracies.append(compute_accuracy(outputs, labels))
    history_buggy['val_accuracy'].append(sum(batch_accuracies) / len(batch_accuracies))

plt.figure(figsize=(8, 4))
plt.plot(history_buggy['epoch'], history_buggy['train_accuracy'], label='Train Accuracy (buggy)')
plt.plot(history_buggy['epoch'], history_buggy['val_accuracy'],   label='Validation Accuracy')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.title('Buggy Plot (val > train incorrectly)')
plt.legend(); plt.grid(True); plt.show()

---
## ✅ QUESTION 4 — Accuracy Plot Analysis

**Q4a. From the plot, are 5 epochs enough or should training continue longer?**

Looking at the plot: if both the train and validation accuracy curves are still **increasing** at epoch 5 without plateauing, the model would benefit from more epochs. If they flatten out, 5 may be sufficient. For the 8×8 MNIST with a simple model, accuracy typically plateaus quickly — more epochs with a larger image (28×28) would be more beneficial.

**Q4b. Why is validation accuracy higher than training accuracy? (Bug explanation and fix)**

**Root cause:** In the original code, training accuracy is measured *during* training (while the model is still updating its weights batch-by-batch). Early batches in each epoch use a *weaker version* of the model, so the average accuracy is pulled down. Validation accuracy is measured *after* the full epoch with the final (best) weights.

**Fix:** Compute training accuracy in a **separate evaluation pass** after each epoch completes — using `model.eval()` and `torch.no_grad()`, exactly as done for validation.

In [ ]:
# prompt: Fix the training loop so that train accuracy is computed after each full epoch
# in eval mode (same conditions as validation), making the train vs val comparison fair.
# Modification: Added a separate evaluation pass over train_loader after each epoch.
# This ensures both curves reflect the same model state (end-of-epoch weights).

num_epochs = 5
history_fixed = {'epoch': [], 'train_accuracy': [], 'val_accuracy': []}

model_fixed = SimpleNN(input_size, hidden_size, num_classes).to(device)
optimizer_fixed = optim.Adam(model_fixed.parameters(), lr=0.001)

for epoch in range(num_epochs):

    # --- Training phase (weights update only, no accuracy tracking) ---
    model_fixed.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model_fixed(images)
        loss = criterion(outputs, labels)
        optimizer_fixed.zero_grad()
        loss.backward()
        optimizer_fixed.step()

    # --- Evaluate on TRAIN set AFTER full epoch (FIX: eval mode, no_grad) ---
    model_fixed.eval()
    batch_accuracies = []
    with torch.no_grad():
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_fixed(images)
            batch_accuracies.append(compute_accuracy(outputs, labels))
    history_fixed['train_accuracy'].append(sum(batch_accuracies) / len(batch_accuracies))

    # --- Evaluate on VALIDATION/TEST set ---
    batch_accuracies = []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_fixed(images)
            batch_accuracies.append(compute_accuracy(outputs, labels))
    history_fixed['val_accuracy'].append(sum(batch_accuracies) / len(batch_accuracies))

    history_fixed['epoch'].append(epoch)
    print(f"Epoch {epoch+1}: Train={history_fixed['train_accuracy'][-1]:.4f}  "
          f"Val={history_fixed['val_accuracy'][-1]:.4f}")

# --- Plot ---
plt.figure(figsize=(8, 4))
plt.plot(history_fixed['epoch'], history_fixed['train_accuracy'], label='Train Accuracy (fixed)')
plt.plot(history_fixed['epoch'], history_fixed['val_accuracy'],   label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Train vs Validation Accuracy (Fixed — train should be ≥ val)')
plt.legend()
plt.grid(True)
plt.show()

print("\nAfter fix: train accuracy should be >= validation accuracy (expected behaviour).")

---
## Summary of Changes Made

| Question | Change |
|----------|--------|
| Q1 | Added markdown cell with answers + verification code cell |
| Q2 | Added `TwoLayerNN` class with second hidden layer; answered Q2a–d in markdown |
| Q3 | Added `train_and_time()` function comparing `num_workers=0` vs `num_workers=2` |
| Q4 | Kept original buggy loop for comparison; added fixed loop with separate eval pass for train accuracy |